In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Находим корень проекта (поднимаемся на уровень выше папки notebooks)
root = Path.cwd().parent
sys.path.append(str(root))
sys.path.append(r"/trinity/home/asma.benachour/BrainBERT")


from utils.analysis_pipeline import (
    load_and_preprocess,
    create_epochs,
    save_epochs,
    plot_epochs_images
)
from utils.config import ch_to_keep, best_ch_by_power
import models
import torch
from omegaconf import OmegaConf

%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats


import mne
# from mne_bids import (
#     BIDSPath,
#     find_matching_paths,
#     get_entity_vals,
#     read_raw_bids,
# )

from scipy.signal import stft
from intervaltree import Interval, IntervalTree
from tqdm import tqdm

In [5]:
def get_stft(x, fs, clip_fs=-1, normalizing=None, **kwargs):
    f, t, Zxx = signal.stft(x, fs, **kwargs)
   
    Zxx = Zxx[:clip_fs]
    f = f[:clip_fs]

    Zxx = np.abs(Zxx)
    clip = 5 #To handle boundary effects
    if normalizing=="zscore":
        Zxx = Zxx #[:,clip:-clip]
        Zxx = stats.zscore(Zxx, axis=-1)
        t = t #[clip:-clip]
    elif normalizing=="baselined":
        Zxx = baseline(Zxx[:,clip:-clip])
        t = t[clip:-clip]
    elif normalizing=="db":
        Zxx = np.log2(Zxx[:,clip:-clip])
        t = t[clip:-clip]

    if np.isnan(Zxx).any():
        import pdb; pdb.set_trace()

    return f, t, Zxx

In [3]:
def annotate_full_stft(
    edf_path: str,
    ch_to_keep: dict,
    subject: str,
    mapping: dict,   # annotation → class_id (пока не используем здесь)
):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    

    ch_to_drop = [ch for ch in raw.ch_names if ch not in ch_to_keep.get(subject)]
    if ch_to_drop:
        raw.drop_channels(ch_to_drop)
    ch_names = raw.info['ch_names']

    sfreq = float(raw.info['sfreq'])
    data  = raw.get_data()
    rec_end = data.shape[1] / sfreq

    ann = raw.annotations

    # Сортируем события по времени и извлекаем имена
    onsets = np.asarray(ann.onset, dtype=float)
    descs = np.array([str(d) for d in ann.description], dtype=np.str_)
    order = np.argsort(onsets)
    onsets = onsets[order]
    descs = descs[order]

    # STFT (использую твой get_stft)
    nperseg, noverlap = 400, 350
    freqs, times, Zxx = get_stft(data, sfreq, clip_fs=40, nperseg=nperseg, noverlap=noverlap, normalizing="zscore", return_onesided=True)

    labels = np.empty(times.shape, dtype='<U64')

    for i, t in enumerate(times):
        # 1) Левый край до первого события → State record
        if t < onsets[0]:
            labels[i] = np.str_('State record')
            continue

        # 2) Правый край → последняя метка
        if t >= rec_end:
            labels[i] = descs[-1]
            continue

        # 3) В середине – ищем ближайший onset слева
        idx = np.searchsorted(onsets, t, side='right') - 1

        if idx < 0:
            labels[i] = np.str_('State record')
        else:
            labels[i] = descs[idx]

    # events_df для удобства
    import pandas as pd
    dur = np.diff(np.r_[onsets, rec_end])
    events_df = pd.DataFrame({"onset": onsets, "duration": dur, "desc": descs})

    return Zxx, freqs, times, labels, events_df, ch_names


In [ ]:
def build_model(cfg):
    ckpt_path = cfg.upstream_ckpt
    init_state = torch.load(ckpt_path, map_location='cpu', weights_only=False)  # ✅
    upstream_cfg = init_state["model_cfg"]
    upstream = models.build_model(upstream_cfg)
    upstream.load_state_dict(init_state["model"], strict=False)
    return upstream

In [ ]:
def load_model_weights(model, states, multi_gpu):
    if multi_gpu:
        model.module.load_weights(states)
    else:
        model.load_weights(states)

In [ ]:
def extract_trials(spec, labels, ignore="State record"):
    # spec: (channels, time, freqs)
    # labels: (time,) строки
    labels = np.array(labels)

    trials = []
    trial_labels = []

    start = 0
    current = labels[0]

    for i in range(1, len(labels)):
        if labels[i] != current:
            end = i
            if current != ignore:
                trials.append(spec[:, start:end, :])  # (C, seg_len, 40)
                trial_labels.append(current)
            current = labels[i]
            start = i

    # последний отрезок
    if current != ignore:
        trials.append(spec[:, start:, :])
        trial_labels.append(current)

    return trials, trial_labels


In [ ]:
def select_embeddings(embeddings, mode="concrete", i=0):
    """
    embeddings:
      - (N, B, 768)
      - (N, B, T, 768)

    Возвращает:
      - concrete:
           (N, 768)        если 3D
           (N, T, 768)     если 4D
      - mean_rest:
           (N, 768)        если 3D
           (N, T, 768)     если 4D
    """

    # определяем число измерений
    ndim = embeddings.ndim

    if mode == "concrete":
        if ndim == 3:
            # (N, B, 768) → (N, 768)
            return embeddings[:, i, :]
        elif ndim == 4:
            # (N, B, T, 768) → (N, T, 768)
            return embeddings[:, i, :, :]
        else:
            raise ValueError("embeddings must be 3D or 4D")

    elif mode == "mean_rest":
        if ndim == 3:
            # (N, B, 768) → (N, 768)
            return embeddings[:, 1:, :].mean(dim=1)
        elif ndim == 4:
            # (N, B, T, 768) → (N, T, 768)
            return embeddings[:, 1:, :, :].mean(dim=1)
        else:
            raise ValueError("embeddings must be 3D or 4D")

    else:
        raise ValueError("mode must be 'concrete' or 'mean_rest'")


In [ ]:
def create_clip_and_mask(clip):        
        # clip: (C, T, 40)
        # средний по каналам (или можно canal attention позже)
        # clip = clip.mean(dim=0, keepdim=True)  # → (1, T, 40)
        mean_chan = clip.mean(dim=0, keepdim=True)   # (1, T, 40)
        clip_2 = torch.cat([mean_chan, clip], dim=0) # (1 + C, T, 40) → (8, T, 40)
        B, T, F = clip_2.shape   # B = 8
        mask = torch.zeros((B, T), dtype=torch.bool)
        return clip_2, mask

In [6]:
# # 1️⃣ Загружаем весь EEG
subject= 's11'

edffile = f"../../PirogovDATA/2025-09-05_s11/session_1/NeoRec_2025-09-05_21-04-18.edf"
mapping = {np.str_('LSL finger_flextion_index'): 1,
        np.str_('LSL finger_flextion_middle'): 2,
        np.str_('LSL finger_flextion_pinky'): 3,
        np.str_('LSL finger_flextion_ring'): 4,
        np.str_('LSL fist_clenching'): 5,
        np.str_('LSL index_middle_pinch'): 6,
        np.str_('LSL index_pinch'): 7,
        np.str_('LSL middle_pinch'): 8,
        np.str_('LSL open_hand'): 9,
        np.str_('LSL thumb_flextion'): 10,
        np.str_('State record'): 11}
spec, freqs, times, labels, events_df, ch_names = annotate_full_stft(
    edf_path=edffile, ch_to_keep=best_ch_by_power, subject=subject, mapping=mapping
)

print("Форма спектра:", spec.shape)
print("Форма временного вектора:", times.shape)
print("Форма меток:", labels.shape)
print(events_df.head())

Форма спектра: (7, 201, 11441)
Форма временного вектора: (11441,)
Форма меток: (11441,)
   onset  duration                       desc
0  0.797     0.291               State record
1  1.088     2.439              LSL open_hand
2  3.527     1.397         LSL thumb_flextion
3  4.924     3.006              LSL open_hand
4  7.930     1.401  LSL finger_flextion_index


In [7]:
ch_names

['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']

In [ ]:
ckpt_path = "../../../../pretrained_weights/stft_large_pretrained.pth"
cfg = OmegaConf.create({"upstream_ckpt": ckpt_path})
model = build_model(cfg)
model.to('cpu')
init_state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
load_model_weights(model, init_state['model'], False)

In [ ]:
spec.shape

In [ ]:
labels

In [ ]:
# spec: (7, 201, 11441)
# Оставляем только первые 40 частот
spec = spec[:, :40, :]   # → (7, 40, 11441)

# Переставляем оси в формат (batch=channels, seq_len=time, feature_dim=freqs)
spec = torch.tensor(spec, dtype=torch.float32).permute(0, 2, 1)
# теперь spec.shape == (7, 11441, 40)


In [ ]:
trials, trial_labels = extract_trials(spec, labels)
print(len(trials), "трайалов найдено")

In [ ]:
import numpy as np
from collections import defaultdict

durations = defaultdict(list)

for clip, lbl in zip(trials, trial_labels):
    # clip has shape (C, T_seg, 40)
    seg_len = clip.shape[1]
    durations[lbl].append(seg_len)

for lbl, vals in durations.items():
    vals = np.array(vals)
    print(f"{lbl:25s} mean = {vals.mean():.1f} bins, std = {vals.std():.1f}, min={vals.min()}, max={vals.max()}, n={len(vals)}")


In [ ]:
lengths = [clip.shape[1] for clip, lbl in zip(trials, trial_labels)]
min_len = min(lengths)
print("Минимальная длина:", min_len)
def crop_to_min_len(clip, target_len):
    # clip: (C, T, 40)
    return clip[:, :target_len, :]

trials_cropped = [crop_to_min_len(c, min_len) for c in trials]


In [ ]:
target_class = "LSL open_hand"   # ← можно поменять на любой
filtered_target_trials = []
filtered_against_trials = []
binary_labels = []

for clip, label in zip(trials_cropped, trial_labels):
    if label == "State record":
        continue  # исключаем фон полностью
    if label == target_class:
        filtered_target_trials.append(clip)
        binary_labels.append(1)
    else:
        filtered_against_trials.append(clip)
        binary_labels.append(0)

trials =  [*filtered_target_trials, *filtered_against_trials]

labels_bin = np.array(binary_labels, dtype=int)

print(f"Всего трайалов: {len(trials)}, классов 'open_hand': {sum(labels_bin)}, остальных: {len(labels_bin)-sum(labels_bin)}")


## Testing and visualising

In [ ]:
embeddings_target = []
embeddings_against = []

for target in filtered_target_trials:
    clip, mask = create_clip_and_mask(target)
    with torch.no_grad():
        emb_t = model(clip, mask, intermediate_rep=True)   # (B, T, 768)
    embeddings_target.append(emb_t)

for against in filtered_against_trials:
    clip, mask = create_clip_and_mask(against)
    with torch.no_grad():
        emb_a = model(clip, mask, intermediate_rep=True)   # (B, T, 768)
    # emb = out.mean(dim=1)  # усреднение по времени → (B, 1, 768)
    embeddings_against.append(emb_a)

embeddings_against = torch.stack(embeddings_against, dim=0) # (N, B, 768)
embeddings_target = torch.stack(embeddings_target, dim=0) # (N, B, 768)

In [ ]:
# channel_modes = {
#     "mean_spec": ("concrete", 0),
#     "Fp1":       ("concrete", 1),
#     "Fpz":       ("concrete", 2),
#     "Fp2":       ("concrete", 3),
#     "F7":        ("concrete", 4),
#     "F3":        ("concrete", 5),
#     "F8":        ("concrete", 6),
#     "Tp7":       ("concrete", 7),
#     "mean_emb":  ("mean_rest", None)
# }
channel_modes = {
    "mean_spec": ("concrete", 0),
    "Fp1":       ("concrete", 1),
    "C4":        ("concrete", 2),
    "Tp7":       ("concrete", 3),
    "P3":        ("concrete", 4),
    "Oz":        ("concrete", 5),
    "mean_emb":  ("mean_rest", None)
}

In [ ]:
# for name, (mode, idx) in channel_modes.items():
#     a_sel = select_embeddings(embeddings_against, mode=mode, i=idx)   # (N, T, 768)
#     t_sel = select_embeddings(embeddings_target, mode=mode, i=idx)   # (N, T, 768)
# порядок строк соблюдаем
channel_order = list(channel_modes.keys())

# собираем усреднённые по N матрицы
A_maps = []   # (T,768) для against
T_maps = []   # (T,768) для target

for name, (mode, idx) in channel_modes.items():
    a_sel = select_embeddings(embeddings_against, mode=mode, i=idx)  # (N, T, 768)
    t_sel = select_embeddings(embeddings_target, mode=mode, i=idx)  # (N, T, 768)

    # если 3D (N,768) → добавим фиктивную ось времени
    if a_sel.ndim == 2:   # (N,768)
        a_sel = a_sel.unsqueeze(1)   # (N,1,768)
        t_sel = t_sel.unsqueeze(1)   # (N,1,768)

    # усреднение по N
    A_maps.append(a_sel.mean(dim=0))  # (T,768)
    T_maps.append(t_sel.mean(dim=0))  # (T,768)

# преобразуем в numpy
A_maps = [m.detach().cpu().numpy() for m in A_maps]
T_maps = [m.detach().cpu().numpy() for m in T_maps]

# общий color range
all_vals = np.concatenate([m.ravel() for m in A_maps + T_maps])
vmin, vmax = np.nanpercentile(all_vals, [2, 98])

# оси
rows = len(channel_order)
T = A_maps[0].shape[0]
F = A_maps[0].shape[1]

extent = [0, T, 0, F]   # x = время, y = фичи (768)

fig = plt.figure(figsize=(10, 3 * rows))
gs = fig.add_gridspec(rows, 3, width_ratios=[1, 1, 0.05], wspace=0.3)

for i, name in enumerate(channel_order):
    ax_left  = fig.add_subplot(gs[i, 0])
    ax_right = fig.add_subplot(gs[i, 1])

    im = ax_left.imshow(A_maps[i].T, aspect='auto', origin='lower',
                        extent=extent, vmin=vmin, vmax=vmax)
    ax_left.set_ylabel(name)
    ax_left.set_title("against")
    ax_left.set_xlabel("T")
    ax_left.set_yticks([])

    ax_right.imshow(T_maps[i].T, aspect='auto', origin='lower',
                    extent=extent, vmin=vmin, vmax=vmax)
    ax_right.set_title("target")
    ax_right.set_xlabel("T")
    ax_right.set_yticks([])

# цветовая шкала — отдельная колонка
cax = fig.add_subplot(gs[:, 2])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("embedding value")

plt.tight_layout()
plt.show()

# Decoding

In [8]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

In [9]:
def run_cv(X, y, name):
    print(f"----starting running {name}----")
    # X: numpy (N, 768), y: numpy (N,)
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    class MLP(nn.Module):
        def __init__(self, input_dim=40, hidden_dim=1024, num_classes=2):
            super().__init__()
            layers = []
            for _ in range(50):
                layers += [
                    nn.Linear(input_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(inplace=True),
                    nn.Dropout(0.1)
                ]
                input_dim = hidden_dim
            layers.append(nn.Linear(hidden_dim, num_classes))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    def train_one_split(train_idx, val_idx):
        model = MLP().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()

        X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
        X_val, y_val = X[val_idx].to(device), y[val_idx].to(device)

        train_losses = []
        val_losses = []

        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            pred = model(X_train)
            loss = loss_fn(pred, y_train)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                pred_val = model(X_val)
                val_loss = loss_fn(pred_val, y_val).item()
                val_losses.append(val_loss)

        pred_val = pred_val.argmax(dim=1)
        acc = accuracy_score(y_val.cpu(), pred_val.cpu())
        f1 = f1_score(y_val.cpu(), pred_val.cpu(), average="binary")

        return acc, f1, train_losses, val_losses

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    all_train_losses = []
    all_val_losses = []
    accs = []
    f1s = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        acc, f1, train_losses, val_losses = train_one_split(train_idx, val_idx)
        accs.append(acc)
        f1s.append(f1)
        all_train_losses.append(train_losses)
        all_val_losses.append(val_losses)
        print(f"Fold {fold+1}: ACC={acc:.3f}, F1={f1:.3f}")

    # усреднение лоссов по фолдам
    train_loss_mean = np.mean(all_train_losses, axis=0)
    val_loss_mean = np.mean(all_val_losses, axis=0)

    print("\n► FINAL MEAN RESULTS")
    print(f"ACC = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"F1  = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")

    return train_loss_mean, val_loss_mean

In [10]:
def show(train_loss_mean_array, val_loss_mean_array, name_array, X = 3, Y = 3):
    
    fig, axes = plt.subplots(Y, X,
                         figsize=(8*X, 5*Y),
                         squeeze=False)
    fig.tight_layout()
    
    for i in range(Y):
        for j in range(X):
            try:           
                # график
                axes[i, j].plot(train_loss_mean_array[X*i+j], label="Train loss")
                axes[i, j].plot(val_loss_mean_array[X*i+j], label="Val loss")
                axes[i, j].set_xlabel("Epoch")
                axes[i, j].set_ylabel("Loss")
                axes[i, j].set_title(f"Cross-Validated Training / Validation Loss over {name_array[X*i+j]}")
                axes[i, j].legend()
                axes[i, j].grid(True)
            except Exception as e:
                print(e)
    fig.show()

In [ ]:
embeddings = []

for clip in trials:
    # clip: (C, T, 40)
    # средний по каналам (или можно canal attention позже)
    # clip = clip.mean(dim=0, keepdim=True)  # → (1, T, 40)
    
    mean_chan = clip.mean(dim=0, keepdim=True)   # (1, T, 40)
    clip_2 = torch.cat([mean_chan, clip], dim=0) # (1 + C, T, 40) → (8, T, 40)

    # B, T, F = clip_2.shape   # B = 8

    # mask = torch.zeros((B, T), dtype=torch.bool)
    # with torch.no_grad():
    #     out = model(clip_2, mask, intermediate_rep=True)   # (B, T, 768)
    # emb = out.mean(dim=1)  # усреднение по времени → (1, 768)

    # embeddings.append(emb)

    clip_2 = clip_2.mean(dim=1)  # усреднение по времени → (1, 768)
    embeddings.append(clip_2)


In [ ]:
clip.shape

In [ ]:
embeddings = torch.stack(embeddings, dim=0)   # (N, B, 768)

In [ ]:
embeddings.shape

In [ ]:
# X = select_embeddings(embeddings, mode="mean_rest")
# y = np.array(labels_bin, dtype=int)


In [ ]:
# X = torch.cat(embeddings, dim=0)   # (N_trials, 768)
# y = np.array(labels_bin)         # (N_trials,)

In [ ]:
channel_modes = {
    "mean_spec": ("concrete", 0),
    "Fp1":       ("concrete", 1),
    "Fpz":       ("concrete", 2),
    "Fp2":       ("concrete", 3),
    "F7":        ("concrete", 4),
    "F3":        ("concrete", 5),
    "F8":        ("concrete", 6),
    "Tp7":       ("concrete", 7),
    # "mean_emb":  ("mean_rest", None)
}
y = np.array(labels_bin, dtype=int)

In [ ]:

train_loss_mean_array, val_loss_mean_array = [], []
for name, (mode, idx) in channel_modes.items():
    X_sel = select_embeddings(embeddings, mode=mode, i=idx)   # (N, 768)
    train_loss_mean, val_loss_mean = run_cv(X_sel, y, name)
    train_loss_mean_array.append(train_loss_mean)
    val_loss_mean_array.append(val_loss_mean)



In [ ]:
show(train_loss_mean_array, val_loss_mean_array, list(channel_modes.keys()))